## ANN Replication: Stage B
- training network (training.py) (stage B)

In [1]:
import sys
from pathlib import Path
import numpy as np

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

DATA_NPZ = ROOT / "data/processed/prepared.npz"
MODEL_DIR = ROOT / "models/ann_baseline" 
MODEL_DIR.mkdir(parents=True, exist_ok=True)

import hashlib
import json

from sandwich.ann.network import LAYERS, N_PARAMS, load_split, mse, predict, r_squared
from sandwich.ann.training import TRAINLM, jacobian, train_lm

data = load_split(DATA_NPZ)
start = np.load(MODEL_DIR / "w0.npz")
w0, SEED = start["w0"], int(start["seed"])
print(f"starting weights from 05a (seed {SEED}), initial train MSE {mse(w0, data['X_train'], data['Y_train']):.3e}") 


starting weights from 05a (seed 42), initial train MSE 3.114e+00


In [2]:
rng_check = np.random.default_rng(0)
X_chk = data["X_train"][:5]
J_chk, _ = jacobian(w0, X_chk)
print("Jacobian shape for 5 designs:", J_chk.shape, " (5 designs x 8 outputs, 1287 weights)")

eps = 1e-6
worst = 0.0
for j in rng_check.choice(N_PARAMS, 50, replace=False):
    step = np.zeros(N_PARAMS)
    step[j] = eps
    fd = (predict(w0 + step, X_chk) - predict(w0 - step, X_chk)).ravel() / (2 * eps)
    worst = max(worst, np.abs(fd - J_chk[:, j]).max())
print(f"largest difference, analytic vs. finite difference: {worst:.2e}")
assert worst < 1e-6, "Jacobian does not match finite differences"


Jacobian shape for 5 designs: (40, 1287)  (5 designs x 8 outputs, 1287 weights)
largest difference, analytic vs. finite difference: 8.18e-10


In [3]:
settings = dict(TRAINLM)

for k, v in settings.items():
    print(f"{k:<9} {v}")

epochs    1000
goal      0.0
max_fail  6
min_grad  1e-07
mu        0.001
mu_dec    0.1
mu_inc    10.0
mu_max    10000000000.0


In [4]:
best, history, stop_reason, train_seconds = train_lm(w0, data, settings)
w_best = best["w"]

print(f"\nstopped: {stop_reason} after {int(history.epoch.iloc[-1])} epochs ({train_seconds / 60:.1f} min)")
print(f"best validation epoch: {best['epoch']}")
print(f"MSE at best epoch   train {mse(w_best, data['X_train'], data['Y_train']):.3e}   "
      f"val {mse(w_best, data['X_val'], data['Y_val']):.3e}   "
      f"test {mse(w_best, data['X_test'], data['Y_test']):.3e}")
print("paper (Sec. 5.1):    best MSE 1.3e-05 after 639 epochs")

epoch   25 train 3.292e-05 val 3.468e-05 test 3.768e-05 mu 1e-01 (51 s
epoch   50 train 1.456e-05 val 1.625e-05 test 1.647e-05 mu 1e-01 (106 s
epoch   75 train 6.507e-06 val 6.934e-06 test 7.326e-06 mu 1e-02 (159 s
epoch  100 train 2.944e-06 val 3.005e-06 test 3.295e-06 mu 1e-02 (217 s
epoch  125 train 2.215e-06 val 2.245e-06 test 2.451e-06 mu 1e-02 (279 s
epoch  150 train 1.695e-06 val 1.745e-06 test 1.879e-06 mu 1e-02 (339 s
epoch  175 train 1.418e-06 val 1.488e-06 test 1.574e-06 mu 1e-02 (401 s
epoch  200 train 1.250e-06 val 1.350e-06 test 1.371e-06 mu 1e-02 (464 s
epoch  225 train 1.119e-06 val 1.239e-06 test 1.228e-06 mu 1e-02 (525 s
epoch  250 train 1.004e-06 val 1.122e-06 test 1.117e-06 mu 1e-02 (594 s
epoch  275 train 9.049e-07 val 1.014e-06 test 1.025e-06 mu 1e-02 (663 s
epoch  300 train 8.305e-07 val 9.326e-07 test 9.527e-07 mu 1e-02 (732 s
epoch  325 train 6.469e-07 val 7.184e-07 test 7.318e-07 mu 1e-03 (796 s
epoch  350 train 3.632e-07 val 4.087e-07 test 4.392e-07 mu 1e-03 

In [5]:
np.savez(MODEL_DIR / "weights.npz", w=w_best, layer=np.array(LAYERS))
history.to_csv(MODEL_DIR / "history.csv", index=False)

parts = {"training": "train", "validation": "val", "test": "test"}
meta = {
    "architecture": "9-28-27-8, tansig hidden, purelin output",
    "initialisation": "Nguyen-Widrow (hidden), uniform [-1, 1] (output)",
    "seed": SEED,
    "trainlm_settings": settings,
    "stop_reason": stop_reason,
    "epochs_run": int(history.epoch.iloc[-1]),
    "best_epoch": int(best["epoch"]),
    "train_minutes": round(train_seconds / 60, 2),
    "mse": {name: mse(w_best, data[f"X_{p}"], data[f"Y_{p}"]) for name, p in parts.items()},
    "r2_pooled": {name: float(r_squared(data[f"Y_{p}"], predict(w_best, data[f"X_{p}"])))
                  for name, p in parts.items()},
    "data_file": str(DATA_NPZ.relative_to(ROOT)),
    "data_md5": hashlib.md5(DATA_NPZ.read_bytes()).hexdigest(),
}

with open(MODEL_DIR / "training_meta.json", "w") as f:
    json.dump(meta, f, indent=2)
print(json.dumps(meta, indent=2))

{
  "architecture": "9-28-27-8, tansig hidden, purelin output",
  "initialisation": "Nguyen-Widrow (hidden), uniform [-1, 1] (output)",
  "seed": 42,
  "trainlm_settings": {
    "epochs": 1000,
    "goal": 0.0,
    "max_fail": 6,
    "min_grad": 1e-07,
    "mu": 0.001,
    "mu_dec": 0.1,
    "mu_inc": 10.0,
    "mu_max": 10000000000.0
  },
  "stop_reason": "maximum epochs reached",
  "epochs_run": 1000,
  "best_epoch": 1000,
  "train_minutes": 37.0,
  "mse": {
    "training": 5.587643063134147e-08,
    "validation": 6.455561841612472e-08,
    "test": 7.168090351712037e-08
  },
  "r2_pooled": {
    "training": 0.9999981604752591,
    "validation": 0.999997878155344,
    "test": 0.9999977044098661
  },
  "data_file": "data/processed/prepared.npz",
  "data_md5": "2eb932c348db2a57b93070d6050d2413"
}
